# 05. Clustering: particiones (etapa 2) y elección de k (etapa 3)

Aplica tres algoritmos a las tres matrices de distancia del notebook 04 (euclídea, DTW y SBD), para k de 2 a 8. Una configuración principal por algoritmo:

- **Jerárquico** con enlace de Ward.
- **Spectral** con afinidad gaussiana de escala local.
- **PAM** (k-medoides).

La etapa 2 genera las particiones y diagnostica sus tamaños; la etapa 3 lee los perfiles para elegir k. La estabilidad con semillas (etapa 4) va después.

**Entrada:** `chartmetric_pilotos/distancias_clustering.rds` (notebook 04).

In [ ]:
# PAQUETES

library(data.table)
library(cluster)
library(mclust)
library(ggplot2)

# Tipografias de la presentacion (se dibujan con ragg): Berlin Sans FB Demi
# para titulos y DejaVu Sans Mono para el resto (copia en fuentes/)
systemfonts::register_font("LWR Titulo", plain = "C:/Windows/Fonts/BRLNSDB.TTF")
systemfonts::register_font(
  "LWR Mono",
  plain = "fuentes/DejaVuSansMono.ttf",
  bold = "fuentes/DejaVuSansMono-Bold.ttf"
)

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

ARCHIVO_DISTANCIAS <- file.path("chartmetric_pilotos", "distancias_clustering.rds")
ARCHIVO_PARTICIONES <- file.path("chartmetric_pilotos", "particiones_clustering.rds")
FIGURES_DIR <- "figuras"

K_CANDIDATOS <- 2:8

# Jerárquico: enlace de Ward sobre distancias (ward.D2 eleva al
# cuadrado las distancias, como corresponde al criterio de Ward)
ENLACE_JERARQUICO <- "ward.D2"

# Spectral: escala local de la afinidad = distancia de cada artista
# a su k-ésimo vecino más cercano; k-means con varios inicios
VECINO_ESCALA_LOCAL <- 7L
INICIOS_KMEANS <- 25L
N_AUTOVALORES_GUARDADOS <- 15L

ETIQUETAS_DISTANCIA <- c(euclidea = "Euclídea", dtw = "DTW", sbd = "SBD")
ETIQUETAS_ALGORITMO <- c(jerarquico = "Jerárquico (Ward)", espectral = "Spectral", pam = "PAM")

# Semillas: misma función y misma semilla primigenia que el notebook 04
SEMILLA_PRIMIGENIA <- 999199L

primos_seis_cifras <- function() {
  limite <- 999999L
  es_primo <- rep(TRUE, limite)
  es_primo[1] <- FALSE
  for (p in 2:floor(sqrt(limite))) {
    if (es_primo[p]) es_primo[seq(p * p, limite, by = p)] <- FALSE
  }
  candidatos <- which(es_primo)
  candidatos[candidatos >= 100000L]
}

PRIMOS_SEIS_CIFRAS <- primos_seis_cifras()

generar_semillas_primas <- function(n, semilla = SEMILLA_PRIMIGENIA) {
  set.seed(semilla)
  orden <- sample(length(PRIMOS_SEIS_CIFRAS))
  PRIMOS_SEIS_CIFRAS[orden[seq_len(n)]]
}

# Control: las primeras semillas coinciden con las del notebook 04
stopifnot(identical(generar_semillas_primas(3), c(742457L, 895957L, 543139L)))

# Semilla del k-means del spectral (la primera de la serie)
SEMILLA_KMEANS_ESPECTRAL <- generar_semillas_primas(1)

SEMILLA_KMEANS_ESPECTRAL

In [ ]:
# ============================================================
# CARGA DE LAS MATRICES DEL NOTEBOOK 04
# ============================================================

stopifnot("Faltan las matrices: correr el notebook 04" = file.exists(ARCHIVO_DISTANCIAS))

distancias <- readRDS(ARCHIVO_DISTANCIAS)
matrices_distancia <- distancias$matrices
artistas <- distancias$artistas
ids_artistas <- as.character(artistas$artist_id)

stopifnot(
  identical(names(matrices_distancia), names(ETIQUETAS_DISTANCIA)),
  all(vapply(matrices_distancia, function(m) identical(rownames(m), ids_artistas), logical(1)))
)

cat(
  "Matrices generadas:", format(distancias$fecha_generacion), "\n",
  "Artistas:", length(ids_artistas), "\n",
  "Distancias:", paste(names(matrices_distancia), collapse = ", "), "\n",
  "Ventana DTW:", distancias$ventana_dtw_meses, "meses\n"
)

## 2. Jerárquico (Ward)

Une en cada paso los dos grupos que menos aumentan la varianza interna. Usa la matriz directamente y es determinístico: no necesita semilla. El dendrograma es el insumo de la lectura de k (etapa 3).

In [ ]:
# ============================================================
# JERARQUICO: un árbol por distancia y cortes en cada k
# ============================================================

arboles_jerarquicos <- lapply(
  matrices_distancia,
  function(m) hclust(as.dist(m), method = ENLACE_JERARQUICO)
)

particiones_jerarquico <- rbindlist(lapply(names(arboles_jerarquicos), function(distancia) {
  cortes <- cutree(arboles_jerarquicos[[distancia]], k = K_CANDIDATOS)
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    artist_id = ids_artistas,
    algoritmo = "jerarquico",
    distancia = distancia,
    k = k,
    cluster = unname(cortes[ids_artistas, as.character(k)])
  )))
}))

In [ ]:
# ============================================================
# FIGURA: dendrogramas de las tres distancias
# ============================================================

COLORES_DISTANCIA <- c(euclidea = "#1982C4", dtw = "#BC4B51", sbd = "#6A4C93")

grafico_dendrogramas <- function() {
  par(mfrow = c(1, 3), bg = "white", mar = c(2, 4.5, 4, 1), oma = c(0, 0, 2.5, 0), las = 1)
  for (distancia in names(arboles_jerarquicos)) {
    plot(
      as.dendrogram(arboles_jerarquicos[[distancia]]),
      leaflab = "none",
      edgePar = list(col = COLORES_DISTANCIA[[distancia]], lwd = 0.6),
      main = ETIQUETAS_DISTANCIA[[distancia]],
      ylab = "Altura de fusión"
    )
  }
  mtext("Clustering jerárquico (Ward) de 1252 trayectorias", side = 3, outer = TRUE, line = 0.5, cex = 1.1)
}

grafico_dendrogramas()

png(file.path(FIGURES_DIR, "clustering_02_dendrogramas.png"), width = 3600, height = 1500, res = 300)
grafico_dendrogramas()
invisible(dev.off())

## 3. Spectral

Convierte cada distancia en afinidad, `exp(-d² / (σᵢ σⱼ))`, con σᵢ igual a la distancia de cada artista a su 7.° vecino (escala local: se adapta a zonas densas y dispersas). Toma los k primeros autovectores de la afinidad normalizada y agrupa con k-means en ese espacio. Así admite cualquier distancia, incluidas DTW y SBD. El k-means usa la semilla derivada de 999199.

In [ ]:
# ============================================================
# SPECTRAL: afinidad, autovectores y k-means por k
# ============================================================

afinidad_escala_local <- function(m, vecino = VECINO_ESCALA_LOCAL) {
  escala <- apply(m, 1, function(fila) sort(fila)[vecino + 1])
  afinidad <- exp(-m^2 / outer(escala, escala))
  diag(afinidad) <- 0
  afinidad
}

# Autovectores de la afinidad normalizada D^(-1/2) A D^(-1/2):
# sus mayores autovalores corresponden a los menores del laplaciano
descomposicion_espectral <- function(m) {
  afinidad <- afinidad_escala_local(m)
  grado <- rowSums(afinidad)
  normalizada <- afinidad / sqrt(outer(grado, grado))
  eigen(normalizada, symmetric = TRUE)
}

descomposiciones <- lapply(matrices_distancia, descomposicion_espectral)

agrupar_espectral <- function(descomposicion, k) {
  incrustacion <- descomposicion$vectors[, seq_len(k), drop = FALSE]
  incrustacion <- incrustacion / sqrt(rowSums(incrustacion^2))
  set.seed(SEMILLA_KMEANS_ESPECTRAL)
  kmeans(incrustacion, centers = k, nstart = INICIOS_KMEANS, iter.max = 100)$cluster
}

particiones_espectral <- rbindlist(lapply(names(descomposiciones), function(distancia) {
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    artist_id = ids_artistas,
    algoritmo = "espectral",
    distancia = distancia,
    k = k,
    cluster = agrupar_espectral(descomposiciones[[distancia]], k)
  )))
}))

# Autovalores guardados para el eigengap de la etapa 3
autovalores_espectral <- rbindlist(lapply(names(descomposiciones), function(distancia) data.table(
  distancia = distancia,
  orden = seq_len(N_AUTOVALORES_GUARDADOS),
  autovalor = descomposiciones[[distancia]]$values[seq_len(N_AUTOVALORES_GUARDADOS)]
)))

# Sanity check: el primer autovalor de la afinidad normalizada es 1
stopifnot(all(abs(autovalores_espectral[orden == 1, autovalor] - 1) < 1e-8))

## 4. PAM (k-medoides)

Elige k artistas reales como centros (medoides) y asigna cada artista al más cercano. Usa la matriz directamente y es determinístico (fases BUILD y SWAP): no necesita semilla. El medoide sirve para contar cada perfil con un caso.

In [ ]:
# ============================================================
# PAM: una corrida por distancia y k
# ============================================================

resultados_pam <- lapply(matrices_distancia, function(m) {
  lapply(setNames(K_CANDIDATOS, K_CANDIDATOS), function(k) pam(as.dist(m), k = k, diss = TRUE))
})

particiones_pam <- rbindlist(lapply(names(resultados_pam), function(distancia) {
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    artist_id = ids_artistas,
    algoritmo = "pam",
    distancia = distancia,
    k = k,
    cluster = unname(resultados_pam[[distancia]][[as.character(k)]]$clustering[ids_artistas])
  )))
}))

medoides_pam <- rbindlist(lapply(names(resultados_pam), function(distancia) {
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    distancia = distancia,
    k = k,
    cluster = seq_len(k),
    artist_id = resultados_pam[[distancia]][[as.character(k)]]$medoids
  )))
}))

## 5. Particiones y tamaños

Las 9 combinaciones en una sola tabla. Los clusters se renumeran por tamaño (1 = el más grande) para leerlos igual en todos los algoritmos. El diagnóstico de tamaños detecta soluciones degeneradas: un cluster que se queda con casi todos los artistas y grupos de unos pocos.

In [ ]:
# ============================================================
# TABLA UNICA DE PARTICIONES, clusters renumerados por tamaño
# ============================================================

particiones <- rbind(particiones_jerarquico, particiones_espectral, particiones_pam)

particiones[
  ,
  cluster := {
    tamanios <- table(cluster)
    orden <- names(sort(tamanios, decreasing = TRUE))
    match(as.character(cluster), orden)
  },
  by = .(algoritmo, distancia, k)
]

# Medoides con la misma numeración que la tabla
medoides_pam <- merge(
  medoides_pam[, .(distancia, k, artist_id)],
  particiones[algoritmo == "pam", .(artist_id, distancia, k, cluster)],
  by = c("artist_id", "distancia", "k")
)
medoides_pam <- merge(medoides_pam, artistas[, .(artist_id = as.character(artist_id), artist_name)], by = "artist_id")

# Sanity check: cada combinación asigna a todos los artistas una sola
# vez y tiene exactamente k clusters
control_particiones <- particiones[
  ,
  .(artistas = .N, unicos = uniqueN(artist_id), clusters = uniqueN(cluster)),
  by = .(algoritmo, distancia, k)
]

stopifnot(
  nrow(control_particiones) == length(ETIQUETAS_ALGORITMO) * length(ETIQUETAS_DISTANCIA) * length(K_CANDIDATOS),
  all(control_particiones$artistas == length(ids_artistas)),
  all(control_particiones$unicos == length(ids_artistas)),
  all(control_particiones$clusters == control_particiones$k)
)

cat("Particiones:", nrow(control_particiones), "(3 algoritmos x 3 distancias x", length(K_CANDIDATOS), "valores de k)\n")

In [ ]:
# ============================================================
# DIAGNOSTICO DE TAMAÑOS
# pct_mayor: % de artistas en el cluster más grande.
# menor: artistas en el cluster más chico.
# ============================================================

tamanios_particiones <- particiones[
  ,
  .N,
  by = .(algoritmo, distancia, k, cluster)
][
  ,
  .(pct_mayor = round(100 * max(N) / sum(N), 1), menor = min(N)),
  by = .(algoritmo, distancia, k)
]

tamanios_particiones[, combinacion := paste(algoritmo, distancia, sep = " | ")]

cat("Porcentaje de artistas en el cluster más grande:\n")
print(dcast(tamanios_particiones, combinacion ~ k, value.var = "pct_mayor"))

cat("\nArtistas en el cluster más chico:\n")
dcast(tamanios_particiones, combinacion ~ k, value.var = "menor")

In [ ]:
# ============================================================
# FIGURA: tamaño del cluster más grande por combinación y k
# ============================================================

datos_figura_tamanios <- copy(tamanios_particiones)
datos_figura_tamanios[
  ,
  `:=`(
    algoritmo_etiqueta = factor(ETIQUETAS_ALGORITMO[algoritmo], levels = ETIQUETAS_ALGORITMO),
    distancia_etiqueta = factor(ETIQUETAS_DISTANCIA[distancia], levels = rev(ETIQUETAS_DISTANCIA))
  )
]

grafico_tamanios <- ggplot(
  datos_figura_tamanios,
  aes(x = factor(k), y = distancia_etiqueta, fill = pct_mayor)
) +
  geom_tile(colour = "white", linewidth = 1) +
  geom_text(aes(label = paste0(round(pct_mayor), "%")), size = 3.6, colour = "#1D1E2C") +
  facet_wrap(~ algoritmo_etiqueta, nrow = 1) +
  scale_fill_gradientn(
    colours = c("#E9F5DB", "#F4A259", "#BC4B51"),
    limits = c(0, 100),
    name = "Artistas en el\ncluster mayor"
  ) +
  labs(
    title = "¿Grupos equilibrados o un cluster dominante?",
    subtitle = "Porcentaje de los 1252 artistas en el cluster más grande, por algoritmo, distancia y k",
    x = "k (cantidad de clusters)",
    y = NULL
  ) +
  theme_minimal(base_size = 13) +
  theme(panel.grid = element_blank(), strip.text = element_text(face = "bold"),
        plot.background = element_rect(fill = "white", colour = NA))

print(grafico_tamanios)

ggsave(
  file.path(FIGURES_DIR, "clustering_03_tamanio_cluster_mayor.png"),
  grafico_tamanios,
  width = 13, height = 4.5, dpi = 300, bg = "white"
)

In [ ]:
# ============================================================
# GUARDADO DE LAS PARTICIONES (archivo local, excluido de git)
# ============================================================

saveRDS(
  list(
    fecha_generacion = Sys.time(),
    archivo_distancias = ARCHIVO_DISTANCIAS,
    configuracion = list(
      k_candidatos = K_CANDIDATOS,
      enlace_jerarquico = ENLACE_JERARQUICO,
      vecino_escala_local = VECINO_ESCALA_LOCAL,
      inicios_kmeans = INICIOS_KMEANS,
      semilla_kmeans_espectral = SEMILLA_KMEANS_ESPECTRAL
    ),
    artistas = artistas,
    particiones = particiones,
    tamanios = tamanios_particiones,
    arboles_jerarquicos = arboles_jerarquicos,
    autovalores_espectral = autovalores_espectral,
    medoides_pam = medoides_pam
  ),
  ARCHIVO_PARTICIONES
)

cat("Particiones guardadas en", ARCHIVO_PARTICIONES, "\n")

## Alcance y limitaciones de la etapa 2

- Una configuración por algoritmo: enlace de Ward (con DTW y SBD, que no son distancias euclídeas, es una aproximación), una sola escala de afinidad en spectral (7.° vecino) y PAM estándar.
- k de 2 a 8: más grupos no se podrían contar en el video.
- Sin GMM: necesita un espacio euclídeo de variables y una matriz DTW no lo es.
- La elección de k y la estabilidad frente a semillas y remuestreo se evalúan en las etapas 3 y 4.

---

# Etapa 3. Lectura de perfiles y elección de k

Tres apoyos, sin silueta (con trayectorias poco separadas da poca información):

- **Eigengap** (spectral): salto grande entre autovalores sucesivos.
- **Saltos del dendrograma** (jerárquico): fusión que une grupos lejanos.
- **Separación entre perfiles**: distancia mínima entre las trayectorias medianas de los clusters. Si al subir k aparece un grupo que repite un perfil ya existente, esa distancia cae: el k nuevo no aporta un perfil distinto.

La decisión final sale de mirar los perfiles (small multiples).

In [ ]:
# ============================================================
# SERIES PARA LOS PERFILES: valor_suavizado del panel preprocesado
# (la misma serie que usaron las distancias del notebook 04)
# ============================================================

series_panel <- fread(
  distancias$archivo_panel,
  select = c("artist_id", "variable", "month", distancias$columna_serie)
)
setnames(series_panel, distancias$columna_serie, "valor")
series_panel[, `:=`(artist_id = as.character(artist_id), month = as.Date(month))]

ETIQUETAS_VARIABLE <- c(
  spotify_followers = "Seguidores Spotify",
  instagram_followers = "Seguidores Instagram",
  spotify_playlist_count = "Playlists Spotify",
  youtube_channel_subscribers = "Suscriptores YouTube",
  youtube_playlist_count = "Playlists YouTube"
)

# Sanity check: mismos artistas que las matrices, 49 meses x 5 variables
stopifnot(
  setequal(unique(series_panel$artist_id), ids_artistas),
  series_panel[, .N, by = artist_id][, all(N == length(ETIQUETAS_VARIABLE) * 49)],
  setequal(unique(series_panel$variable), names(ETIQUETAS_VARIABLE))
)

In [ ]:
# ============================================================
# EIGENGAP (spectral) Y SALTOS DEL DENDROGRAMA (jerárquico)
# eigengap_k: autovalor k menos autovalor k + 1.
# salto_k: altura de la fusión que pasa de k a k - 1 grupos,
# relativa a la fusión siguiente (cuánto más lejos se une).
# ============================================================

eigengap <- autovalores_espectral[
  order(distancia, orden),
  .(k = orden, eigengap = autovalor - shift(autovalor, type = "lead")),
  by = distancia
][k %in% K_CANDIDATOS]

saltos_dendrograma <- rbindlist(lapply(names(arboles_jerarquicos), function(distancia) {
  alturas <- sort(arboles_jerarquicos[[distancia]]$height, decreasing = TRUE)
  data.table(
    distancia = distancia,
    k = K_CANDIDATOS,
    salto_relativo = alturas[K_CANDIDATOS - 1] / alturas[K_CANDIDATOS]
  )
}))

k_sugerido_indicadores <- merge(
  eigengap[, .SD[which.max(eigengap)], by = distancia][, .(distancia, k_eigengap = k)],
  saltos_dendrograma[, .SD[which.max(salto_relativo)], by = distancia][, .(distancia, k_dendrograma = k)],
  by = "distancia"
)

cat("Eigengap por k (spectral):\n")
print(dcast(eigengap[, .(distancia, k, eigengap = round(eigengap, 4))], distancia ~ k, value.var = "eigengap"))
cat("\nSalto relativo del dendrograma por k (jerárquico):\n")
print(dcast(saltos_dendrograma[, .(distancia, k, salto = round(salto_relativo, 2))], distancia ~ k, value.var = "salto"))
cat("\nk sugerido por cada indicador:\n")
k_sugerido_indicadores

In [ ]:
# ============================================================
# PERFILES MEDIANOS Y SEPARACION ENTRE PERFILES
# Perfil de un cluster: mediana de valor_suavizado de sus
# artistas, por variable y mes (49 x 5 valores).
# separacion_minima: distancia euclídea entre los dos perfiles
# más parecidos de la partición, relativa a la distancia euclídea
# mediana entre artistas (1 = tan distintos como dos artistas
# típicos). Se calcula con matrices, sin multiplicar filas.
# ============================================================

# Matriz artista x (variable, mes): una fila por artista, 245 columnas
series_ancho <- dcast(series_panel, artist_id ~ variable + month, value.var = "valor")
matriz_series <- as.matrix(series_ancho[, -1])
rownames(matriz_series) <- series_ancho$artist_id
matriz_series <- matriz_series[ids_artistas, ]

columnas_series <- data.table(columna = colnames(matriz_series))
columnas_series[, `:=`(
  variable = sub("_[0-9]{4}-[0-9]{2}-[0-9]{2}$", "", columna),
  month = as.Date(sub(".*_", "", columna))
)]

# Perfil mediano de cada cluster en cada partición
perfil_mediano_clusters <- function(asignacion) {
  t(vapply(sort(unique(asignacion)), function(g)
    apply(matriz_series[asignacion == g, , drop = FALSE], 2, median), numeric(ncol(matriz_series))))
}

DISTANCIA_MEDIANA_ARTISTAS_EUCLIDEA <- median(matrices_distancia$euclidea[upper.tri(matrices_distancia$euclidea)])

separacion_perfiles <- particiones[
  ,
  {
    asignacion <- cluster[match(ids_artistas, artist_id)]
    .(separacion_minima = min(dist(perfil_mediano_clusters(asignacion))) / DISTANCIA_MEDIANA_ARTISTAS_EUCLIDEA)
  },
  by = .(algoritmo, distancia, k)
]

# Perfiles en formato largo, sólo para las figuras (DTW, k de lectura)
K_LECTURA <- 3:6

perfiles <- particiones[distancia == "dtw" & k %in% K_LECTURA][
  ,
  {
    asignacion <- cluster[match(ids_artistas, artist_id)]
    matriz_perfiles <- perfil_mediano_clusters(asignacion)
    grupos <- sort(unique(asignacion))
    data.table(
      cluster = rep(grupos, times = ncol(matriz_perfiles)),
      artistas = rep(as.vector(table(asignacion)[as.character(grupos)]), times = ncol(matriz_perfiles)),
      variable = rep(columnas_series$variable, each = length(grupos)),
      month = rep(columnas_series$month, each = length(grupos)),
      valor_mediano = as.vector(matriz_perfiles)
    )
  },
  by = .(algoritmo, distancia, k)
]

separacion_perfiles <- merge(separacion_perfiles, tamanios_particiones[, .(algoritmo, distancia, k, menor)],
                             by = c("algoritmo", "distancia", "k"))

# Cluster chico: menos del 2% de los artistas (25), difícil de contar como perfil
UMBRAL_CLUSTER_CHICO <- ceiling(0.02 * length(ids_artistas))

cat("Separación mínima entre perfiles (relativa a la distancia mediana entre artistas):\n")
print(dcast(separacion_perfiles[, .(combinacion = paste(algoritmo, distancia, sep = " | "), k,
                                    separacion = round(separacion_minima, 2))],
            combinacion ~ k, value.var = "separacion"))

cat("\nCombinaciones y k con algún cluster de menos de", UMBRAL_CLUSTER_CHICO, "artistas:\n")
separacion_perfiles[menor < UMBRAL_CLUSTER_CHICO, .(k = paste(k, collapse = ", ")), by = .(algoritmo, distancia)]

In [ ]:
# ============================================================
# FIGURA: separación mínima entre perfiles según k
# Colores propios de esta figura: euclídea en negro, DTW en
# rojo y SBD en gris medio. Leyendas abajo, como zócalo.
# ============================================================

COLORES_DISTANCIA_SEPARACION <- c(euclidea = "#111111", dtw = "#C8102E", sbd = "#8C8C8C")

datos_figura_separacion <- copy(separacion_perfiles)
datos_figura_separacion[
  ,
  `:=`(
    algoritmo_etiqueta = factor(ETIQUETAS_ALGORITMO[algoritmo], levels = ETIQUETAS_ALGORITMO),
    distancia_etiqueta = factor(ETIQUETAS_DISTANCIA[distancia], levels = ETIQUETAS_DISTANCIA),
    cluster_chico = menor < UMBRAL_CLUSTER_CHICO
  )
]

grafico_separacion <- ggplot(datos_figura_separacion, aes(x = k, y = separacion_minima, colour = distancia_etiqueta)) +
  geom_line(linewidth = 1.1) +
  geom_point(aes(shape = cluster_chico), size = 3, stroke = 1.2) +
  facet_wrap(~ algoritmo_etiqueta, nrow = 1) +
  scale_colour_manual(values = setNames(COLORES_DISTANCIA_SEPARACION, ETIQUETAS_DISTANCIA), name = "Distancia") +
  scale_shape_manual(values = c(`FALSE` = 16, `TRUE` = 4), labels = c("No", "Sí"),
                     name = paste0("Cluster de menos de ", UMBRAL_CLUSTER_CHICO, " artistas")) +
  scale_x_continuous(breaks = K_CANDIDATOS) +
  guides(colour = guide_legend(order = 1), shape = guide_legend(order = 2)) +
  labs(
    title = "¿Cuántos perfiles distintos hay?",
    subtitle = paste0(
      "Distancia entre los dos perfiles medianos más parecidos,
",
      "relativa a la distancia típica entre dos artistas"
    ),
    x = "k (cantidad de clusters)",
    y = "Separación mínima entre perfiles"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    strip.text = element_text(face = "bold", size = 12, colour = "#111111"),
    axis.title = element_text(colour = "#4A4A4A", size = 12),
    axis.text = element_text(colour = "#6B6B6B"),
    legend.position = "bottom",
    legend.box = "horizontal",
    legend.box.just = "left",
    legend.spacing.x = unit(28, "pt"),
    legend.title = element_text(size = 11, colour = "#4A4A4A"),
    legend.text = element_text(size = 11, colour = "#4A4A4A"),
    legend.margin = margin(t = 6),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(colour = "#EDEDED"),
    panel.spacing = unit(18, "pt"),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18)
  )

ggsave(file.path(FIGURES_DIR, "clustering_04_separacion_perfiles.png"), grafico_separacion,
       width = 11, height = 6.5, dpi = 300, bg = "white", device = ragg::agg_png)

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "clustering_04_separacion_perfiles.png"))


In [ ]:
# ============================================================
# FIGURAS: perfiles medianos por cluster (small multiples)
# Filas: k; columnas: variables; una línea por cluster, con su
# tamaño en la leyenda. Sólo DTW (distancia principal), k 3 a 6.
# ============================================================

PALETA_CLUSTERS <- c("#1982C4", "#F04A50", "#00A878", "#FF7A00", "#6A4C93", "#E4007C")

grafico_perfiles <- function(algoritmo_elegido, distancia_elegida) {
  datos <- perfiles[algoritmo == algoritmo_elegido & distancia == distancia_elegida & k %in% K_LECTURA]
  datos[, `:=`(
    variable_etiqueta = factor(ETIQUETAS_VARIABLE[variable], levels = ETIQUETAS_VARIABLE),
    k_etiqueta = paste0("k = ", k),
    cluster_etiqueta = factor(cluster)
  )]
  etiquetas_tamanio <- unique(datos[, .(k_etiqueta, cluster_etiqueta, artistas, month = max(month))])

  ggplot(datos, aes(x = month, y = valor_mediano, colour = cluster_etiqueta, group = cluster_etiqueta)) +
    geom_hline(yintercept = 0, colour = "grey80") +
    geom_line(linewidth = 1) +
    facet_grid(k_etiqueta ~ variable_etiqueta, scales = "free_y") +
    scale_colour_manual(values = PALETA_CLUSTERS, name = "Cluster") +
    scale_x_date(date_breaks = "1 year", date_labels = "%Y") +
    labs(
      title = paste0("Perfiles medianos: ", ETIQUETAS_ALGORITMO[[algoritmo_elegido]], " con ",
                     ETIQUETAS_DISTANCIA[[distancia_elegida]]),
      subtitle = "Mediana de log1p centrado y suavizado por cluster (0 = nivel típico del artista). Cluster 1 = el más grande",
      x = NULL,
      y = "log1p centrado"
    ) +
    theme_minimal(base_size = 11) +
    theme(panel.grid.minor = element_blank(), strip.text = element_text(face = "bold"),
          legend.position = "bottom", plot.background = element_rect(fill = "white", colour = NA))
}

tamanios_lectura <- particiones[distancia == "dtw" & k %in% K_LECTURA, .N, by = .(algoritmo, k, cluster)][
  order(algoritmo, k, cluster)][, .(artistas_por_cluster = paste(N, collapse = " / ")), by = .(algoritmo, k)]

for (algoritmo_elegido in names(ETIQUETAS_ALGORITMO)) {
  grafico <- grafico_perfiles(algoritmo_elegido, "dtw")
  print(grafico)
  ggsave(file.path(FIGURES_DIR, paste0("clustering_05_perfiles_", algoritmo_elegido, "_dtw.png")),
         grafico, width = 15, height = 10, dpi = 300, bg = "white")
}

cat("Artistas por cluster (DTW):\n")
tamanios_lectura

In [ ]:
# ============================================================
# FIGURA PARA LA PRESENTACIÓN: los tres perfiles de intensidad
# Spectral con DTW, k = 3. Colores de línea de la paleta de Come
# Together, los mismos que usa el notebook 07 para cada perfil:
# estable amarillo, moderado lila, fuerte celeste.
# Los nombres se asignan por la suba total del perfil
# (último mes menos primero, promedio de las 5 variables).
# ============================================================

perfiles_intensidad <- perfiles[algoritmo == "espectral" & distancia == "dtw" & k == 3]

suba_por_cluster <- perfiles_intensidad[
  ,
  .(suba = valor_mediano[month == max(month)] - valor_mediano[month == min(month)]),
  by = .(cluster, artistas, variable)
][, .(suba_media = mean(suba)), by = .(cluster, artistas)][order(suba_media)]

suba_por_cluster[, perfil := c("Estable", "Moderado", "Fuerte")]
suba_por_cluster[, perfil_etiqueta := paste0(perfil, " (", artistas, ")")]

COLORES_INTENSIDAD <- setNames(c("#C9B800", "#8B6FD6", "#1E8FD8"), suba_por_cluster$perfil_etiqueta)

perfiles_intensidad <- merge(perfiles_intensidad, suba_por_cluster[, .(cluster, perfil_etiqueta)], by = "cluster")
perfiles_intensidad[, `:=`(
  variable_etiqueta = factor(ETIQUETAS_VARIABLE[variable], levels = ETIQUETAS_VARIABLE),
  perfil_etiqueta = factor(perfil_etiqueta, levels = suba_por_cluster$perfil_etiqueta)
)]

# Sanity check: tres perfiles, ordenados de menor a mayor suba
stopifnot(nrow(suba_por_cluster) == 3, !is.unsorted(suba_por_cluster$suba_media))
suba_por_cluster

grafico_perfiles_intensidad <- ggplot(perfiles_intensidad,
                                      aes(x = month, y = valor_mediano, colour = perfil_etiqueta)) +
  geom_hline(yintercept = 0, colour = "#D0D0D0", linewidth = 0.4) +
  geom_line(linewidth = 1.3) +
  facet_wrap(~ variable_etiqueta, ncol = 3) +
  scale_colour_manual(values = COLORES_INTENSIDAD, name = "Perfil (artistas)") +
  scale_x_date(date_breaks = "1 year", date_labels = "%Y") +
  labs(
    title = "Tres perfiles de intensidad",
    subtitle = paste0(
      "Perfil mediano por cluster (spectral con DTW, k = 3).
",
      "0 = el grupo en su propio promedio del período: la pendiente dice cuánto crece"
    ),
    x = NULL,
    y = "log1p centrado"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    strip.text = element_text(face = "bold", size = 12, colour = "#111111", hjust = 0),
    axis.title = element_text(colour = "#4A4A4A", size = 12),
    axis.text = element_text(colour = "#6B6B6B", size = 10),
    legend.position = "inside",
    legend.position.inside = c(0.84, 0.24),
    legend.title = element_text(size = 12, colour = "#4A4A4A"),
    legend.text = element_text(face = "bold", size = 12, colour = "#111111"),
    legend.key.width = unit(34, "pt"),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(colour = "#EDEDED"),
    panel.spacing = unit(18, "pt"),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18)
  )

ggsave(file.path(FIGURES_DIR, "clustering_05b_perfiles_intensidad_k3.png"), grafico_perfiles_intensidad,
       width = 11, height = 7.5, dpi = 300, bg = "white", device = ragg::agg_png)

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "clustering_05b_perfiles_intensidad_k3.png"))


In [ ]:
# ============================================================
# GUARDADO DE LOS INDICADORES DE LA ETAPA 3
# ============================================================

particiones_guardadas <- readRDS(ARCHIVO_PARTICIONES)
particiones_guardadas$etapa_3 <- list(
  eigengap = eigengap,
  saltos_dendrograma = saltos_dendrograma,
  separacion_perfiles = separacion_perfiles,
  perfiles = perfiles
)
saveRDS(particiones_guardadas, ARCHIVO_PARTICIONES)

cat("Indicadores de la etapa 3 agregados a", ARCHIVO_PARTICIONES, "\n")

### ¿Las tres distancias encuentran los mismos grupos?

Acuerdo entre las particiones de k = 3 de cada distancia (índice de Rand ajustado: 1 = misma partición, 0 = acuerdo de azar).

In [ ]:
# ============================================================
# ACUERDO ENTRE DISTANCIAS: ARI entre las particiones de k = 3
# ============================================================

asignacion_k3 <- function(algoritmo_elegido, distancia_elegida) {
  particiones[algoritmo == algoritmo_elegido & distancia == distancia_elegida & k == 3][
    match(ids_artistas, artist_id), cluster]
}

acuerdo_distancias <- rbindlist(lapply(c("espectral", "pam"), function(algoritmo_elegido) {
  data.table(
    algoritmo = algoritmo_elegido,
    euclidea_vs_dtw = round(adjustedRandIndex(asignacion_k3(algoritmo_elegido, "euclidea"), asignacion_k3(algoritmo_elegido, "dtw")), 2),
    dtw_vs_sbd = round(adjustedRandIndex(asignacion_k3(algoritmo_elegido, "dtw"), asignacion_k3(algoritmo_elegido, "sbd")), 2),
    euclidea_vs_sbd = round(adjustedRandIndex(asignacion_k3(algoritmo_elegido, "euclidea"), asignacion_k3(algoritmo_elegido, "sbd")), 2)
  )
}))

acuerdo_distancias

### ¿Estamos leyendo formas?

Sólo en parte. El centrado quita el nivel, pero no la recta de crecimiento, y esa recta explica el 95% de la variación de cada serie (R² mediano de 0,97 en Spotify y YouTube). Los clusters ordenan sobre todo por **cuánto** crece cada artista: la tendencia pesa tanto que tapa la forma.

In [ ]:
# ============================================================
# FIGURA: pendiente anual por cluster (spectral, DTW, k = 3)
# eta2: proporción de la variación de la pendiente media de cada
# artista (promedio de las 5 variables) explicada por el cluster
# ============================================================

pendientes_artista <- series_panel[
  ,
  .(pendiente_anual = 12 * unname(coef(lm(valor ~ seq_along(valor)))[2])),
  by = .(artist_id, variable)
]

clusters_principales <- particiones[distancia == "dtw" & k == 3 & algoritmo %in% c("espectral", "pam"),
                                    .(artist_id, algoritmo, cluster)]

pendientes_cluster <- merge(pendientes_artista, clusters_principales, by = "artist_id", allow.cartesian = TRUE)

eta2_pendiente <- pendientes_cluster[
  ,
  .(pendiente_media = mean(pendiente_anual)),
  by = .(artist_id, algoritmo, cluster)
][
  ,
  .(eta2 = round(summary(lm(pendiente_media ~ factor(cluster)))$r.squared, 2)),
  by = algoritmo
]

# Figura con los nombres y colores de los perfiles (los de la celda
# de la figura de perfiles de intensidad y del notebook 07): relleno
# claro y borde de línea. Arriba las audiencias, abajo los conteos
# en playlists.
COLORES_INTENSIDAD_RELLENO <- c(Estable = "#FFF97A", Moderado = "#C6AEF6", Fuerte = "#47B3F7")
COLORES_INTENSIDAD_LINEA <- c(Estable = "#C9B800", Moderado = "#8B6FD6", Fuerte = "#1E8FD8")

ORDEN_VARIABLES_FIGURA <- c("spotify_followers", "instagram_followers", "youtube_channel_subscribers",
                            "spotify_playlist_count", "youtube_playlist_count")

datos_figura_pendiente <- merge(pendientes_cluster[algoritmo == "espectral"],
                                suba_por_cluster[, .(cluster, perfil)], by = "cluster")
datos_figura_pendiente[, `:=`(
  variable_etiqueta = factor(ETIQUETAS_VARIABLE[variable], levels = ETIQUETAS_VARIABLE[ORDEN_VARIABLES_FIGURA]),
  perfil = factor(perfil, levels = names(COLORES_INTENSIDAD_RELLENO))
)]

# Sanity check: tres perfiles con nombre y las 5 variables en el orden de la figura
stopifnot(!anyNA(datos_figura_pendiente$perfil), !anyNA(datos_figura_pendiente$variable_etiqueta))

# Escala común por fila: audiencias hasta 2, conteos en playlists hasta 1,5.
# Las cajas se calculan con todos los artistas y recién después se recortan
# a la escala: los atípicos que quedan fuera no cambian cuartiles ni medianas.
LIMITES_EJE_Y <- list(
  audiencias = c(-0.1, 2),
  playlists = c(-0.5, 1.5)
)
datos_figura_pendiente[, fila := fifelse(grepl("playlist", variable), "playlists", "audiencias")]
datos_figura_pendiente[, `:=`(
  y_minimo = vapply(fila, function(f) LIMITES_EJE_Y[[f]][1], numeric(1)),
  y_maximo = vapply(fila, function(f) LIMITES_EJE_Y[[f]][2], numeric(1))
)]

cajas_pendiente <- datos_figura_pendiente[
  ,
  {
    cuartiles <- quantile(pendiente_anual, c(0.25, 0.5, 0.75), names = FALSE)
    rango_intercuartil <- cuartiles[3] - cuartiles[1]
    dentro <- pendiente_anual[pendiente_anual >= cuartiles[1] - 1.5 * rango_intercuartil &
                              pendiente_anual <= cuartiles[3] + 1.5 * rango_intercuartil]
    .(inferior = cuartiles[1], mediana = cuartiles[2], superior = cuartiles[3],
      bigote_bajo = max(min(dentro), y_minimo[1]), bigote_alto = min(max(dentro), y_maximo[1]))
  },
  by = .(variable_etiqueta, perfil)
]

atipicos_pendiente <- merge(datos_figura_pendiente, cajas_pendiente, by = c("variable_etiqueta", "perfil"))[
  pendiente_anual < inferior - 1.5 * (superior - inferior) | pendiente_anual > superior + 1.5 * (superior - inferior)
]
atipicos_fuera_de_escala <- atipicos_pendiente[pendiente_anual < y_minimo | pendiente_anual > y_maximo]
atipicos_pendiente <- atipicos_pendiente[pendiente_anual >= y_minimo & pendiente_anual <= y_maximo]

# Puntos invisibles que fijan el rango de cada panel
rango_paneles <- unique(datos_figura_pendiente[, .(variable_etiqueta, y_minimo, y_maximo)])
rango_paneles <- melt(rango_paneles, id.vars = "variable_etiqueta", value.name = "pendiente_anual")

# Sanity check: las cajas quedan dentro de la escala y se informan los atípicos omitidos
stopifnot(cajas_pendiente[, all(inferior >= -0.5 & superior <= 1.5)])
cat("Atípicos fuera de escala (no se dibujan; no cambian las cajas):", nrow(atipicos_fuera_de_escala), "
")
print(atipicos_fuera_de_escala[, .N, by = .(variable_etiqueta, perfil)])

grafico_pendiente_cluster <- ggplot(cajas_pendiente, aes(x = perfil, fill = perfil, colour = perfil)) +
  geom_blank(data = rango_paneles, aes(x = NULL, y = pendiente_anual), inherit.aes = FALSE) +
  geom_hline(yintercept = 0, colour = "#BDBDBD", linewidth = 0.4) +
  geom_boxplot(aes(ymin = bigote_bajo, lower = inferior, middle = mediana, upper = superior, ymax = bigote_alto),
               stat = "identity", width = 0.62, linewidth = 0.7) +
  geom_point(data = atipicos_pendiente, aes(x = perfil, y = pendiente_anual, colour = perfil),
             inherit.aes = FALSE, alpha = 0.35, size = 0.9, show.legend = FALSE) +
  facet_wrap(~ variable_etiqueta, ncol = 3, scales = "free_y") +
  scale_y_continuous(breaks = seq(-0.5, 2, by = 0.5), expand = expansion(mult = 0.02)) +
  scale_fill_manual(values = COLORES_INTENSIDAD_RELLENO, name = "Perfil") +
  scale_colour_manual(values = COLORES_INTENSIDAD_LINEA, name = "Perfil") +
  labs(
    title = "Los clusters ordenan por cuánto crece cada artista",
    subtitle = "Spectral con DTW, k = 3",
    x = NULL, y = "Pendiente anual de log1p
(0,1 ≈ +10% por año; 0,7 ≈ el doble)"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    strip.text = element_text(face = "bold", size = 12, colour = "#111111", hjust = 0),
    axis.title = element_text(colour = "#4A4A4A", size = 12),
    axis.text.x = element_blank(),
    legend.position = "inside",
    legend.position.inside = c(0.84, 0.22),
    legend.title = element_text(size = 12, colour = "#4A4A4A"),
    legend.text = element_text(face = "bold", size = 12, colour = "#111111"),
    legend.key.size = unit(22, "pt"),
    axis.text.y = element_text(colour = "#6B6B6B", size = 10),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank(),
    panel.grid.major.y = element_line(colour = "#EDEDED"),
    panel.spacing = unit(18, "pt"),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18)
  )

ggsave(file.path(FIGURES_DIR, "clustering_06_pendiente_por_cluster.png"), grafico_pendiente_cluster,
       width = 11, height = 7.5, dpi = 300, bg = "white", device = ragg::agg_png)

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "clustering_06_pendiente_por_cluster.png"))

eta2_pendiente


## Lectura preliminar de la etapa 3 (a validar con la estabilidad, etapa 4)

- **k = 3 con spectral o PAM sobre DTW.** Los dos algoritmos encuentran los mismos tres perfiles con tamaños parecidos (spectral 584 / 500 / 168; PAM 583 / 554 / 115):
  1. **Estables:** casi sin crecimiento en Instagram y YouTube, crecimiento leve en Spotify.
  2. **Crecimiento moderado sostenido** en todas las plataformas.
  3. **Crecimiento fuerte:** artistas en expansión, con subas grandes y simultáneas en Spotify, Instagram y YouTube.
- **Qué separan:** niveles de intensidad del crecimiento (pendiente), no formas distintas. La forma se aborda con una segunda representación (notebook 06).
- **Por qué no más grupos:** desde k = 4 la separación mínima entre perfiles cae a la mitad (spectral 0,47 a 0,25; PAM 0,50 a 0,35). El grupo nuevo repite un perfil existente con otra intensidad.
- **Por qué no k = 2:** el eigengap y el dendrograma lo sugieren, pero sólo separa a los de crecimiento fuerte del resto.
- **Euclídea, DTW y SBD:** con el nivel, euclídea y DTW forman casi los mismos grupos (ARI 0,90 con spectral); SBD forma grupos distintos (ARI 0,07) y menos separados (0,27 frente a 0,47). SBD queda como contraste explorado: la pregunta por la forma se responde con la segunda representación (notebook 06).
- **Jerárquico (Ward):** separa primero casos extremos (grupos de 11 a 51 artistas). Sirve para detectar atípicos, no para describir perfiles.
- **A revisar:** desde k = 5 aparece un grupo chico (15 a 40 artistas) con un salto en escalón de suscriptores de YouTube en 2024, posible cambio de canal en la fuente. Las playlists de Spotify crecen igual en todos los clusters: no distinguen perfiles.